# AIS 데이터 분석 (시트별 처리)
부산 실해역 테스트 데이터

In [3]:
import pandas as pd

FILE_PATH = 'AIS_부산 실해역 테스트 데이터3.xlsx'

## 1. 파일 구조 확인

In [4]:
xl = pd.ExcelFile(FILE_PATH)
print('시트 목록:', xl.sheet_names)

시트 목록: ['26.06.09', '26.06.10(~08 30)', '26.06.10(10 33 ~ 15 06)']


In [5]:
# 전체 시트 읽기 (헤더 없음)
sheets = pd.read_excel(FILE_PATH, sheet_name=None, header=None)

for name, df in sheets.items():
    print(f'\n=== 시트: {name} ===')
    print(f'행: {df.shape[0]}, 열: {df.shape[1]}')
    print('샘플:', df.iloc[0].tolist())


=== 시트: 26.06.09 ===
행: 692945, 열: 2
샘플: ['20260609 17:56:42.9433', '!AIVDM,1,1,3,B,36Sf4MPP@Ta>qd`D5PSri9eD21q0,0*52']

=== 시트: 26.06.10(~08 30) ===
행: 991908, 열: 2
샘플: ['20260610 00:00:00.2866', '!AIVDM,1,1,6,A,16S`e90P?w<tSF0l4Q@>41gp1PRL,0*27']

=== 시트: 26.06.10(10 33 ~ 15 06) ===
행: 362776, 열: 2
샘플: ['20260610 11:01:44.6683', '!AIVDM,1,1,7,B,16Sl:g201ia>kv4D5aqKr9OD00SD,0*26']


## 2. 파싱 함수 정의

In [6]:
from pyais import decode

TYPE_1_3_FIELDS = [
    'msg_type', 'repeat', 'mmsi', 'status', 'turn', 'speed',
    'accuracy', 'lon', 'lat', 'course', 'heading', 'second',
    'maneuver', 'raim', 'radio'
]

def decode_msg_type(payload):
    try:
        val = ord(payload[0]) - 48
        if val > 39:
            val -= 8
        return val
    except:
        return None

def parse_vdm_channel(raw_msg):
    try:
        return raw_msg.split('|')[0].split(',')[4]
    except:
        return None

def parse_vsi_fields(vsi):
    try:
        parts = vsi.split(',')
        toa = parts[3]
        return {
            'vsi_ui':       int(parts[1]),
            'vsi_link':     int(parts[2]),
            'vsi_hour':     int(toa[0:2]),
            'vsi_minute':   int(toa[2:4]),
            'vsi_second':   float(toa[4:]),
            'vsi_slot':     int(parts[4]),
            'vsi_strength': int(parts[5]),
            'vsi_snr':      int(parts[6].split('*')[0]),
        }
    except:
        return {k: None for k in ['vsi_ui','vsi_link','vsi_hour','vsi_minute','vsi_second','vsi_slot','vsi_strength','vsi_snr']}

def parse_comm_state(radio, msg_type):
    r = int(radio)
    sync_state = (r >> 17) & 0x3
    if msg_type == 1:
        return {
            'sync_state':     sync_state,
            'slot_timeout':   (r >> 14) & 0x7,
            'sub_message':    r & 0x3FFF,
            'slot_increment': None,
            'num_slots':      None,
            'keep_flag':      None,
        }
    elif msg_type == 3:
        return {
            'sync_state':     sync_state,
            'slot_timeout':   None,
            'sub_message':    None,
            'slot_increment': (r >> 5) & 0x1FFF,
            'num_slots':      (r >> 2) & 0x7,
            'keep_flag':      r & 0x1,
        }
    return {k: None for k in ['sync_state','slot_timeout','sub_message','slot_increment','num_slots','keep_flag']}

def parse_type_1_3(row):
    raw_msg = row['raw_msg']
    vsi     = row['vsi']
    try:
        parts   = [p.encode() for p in raw_msg.split('|')]
        decoded = decode(*parts).asdict()
        result  = {'timestamp': row['timestamp']}
        result['vdm_channel'] = parse_vdm_channel(raw_msg)
        result.update(parse_vsi_fields(vsi))
        result.update({f: decoded.get(f) for f in TYPE_1_3_FIELDS})
        result.update(parse_comm_state(decoded['radio'], decoded['msg_type']))
        result['raw_msg'] = raw_msg
        result['vsi']     = vsi
        return result
    except:
        all_keys = (
            ['timestamp', 'vdm_channel'] +
            ['vsi_ui','vsi_link','vsi_hour','vsi_minute','vsi_second','vsi_slot','vsi_strength','vsi_snr'] +
            TYPE_1_3_FIELDS +
            ['sync_state','slot_timeout','sub_message','slot_increment','num_slots','keep_flag'] +
            ['raw_msg','vsi']
        )
        return {k: None for k in all_keys}

def build_result_df(df):
    """원시 데이터프레임 → VDM/VSI 페어링 + 멀티파트 조립"""
    records    = []
    vdm_buffer = []

    def _parse_vdm(msg):
        p = msg.split(',')
        return {'total': int(p[1]), 'part_num': int(p[2]), 'seq_id': p[3], 'payload': p[5]}

    for _, row in df.iterrows():
        msg = str(row['raw_msg']).strip()
        ts  = row['timestamp']

        if msg.startswith('!'):
            p = _parse_vdm(msg)
            vdm_buffer.append((ts, msg, p))
            if p['part_num'] == p['total']:
                combined_raw = '|'.join(m for _, m, _ in vdm_buffer)
                full_payload = ''.join(fp['payload'] for _, _, fp in vdm_buffer)
                records.append({
                    'timestamp': vdm_buffer[0][0],
                    'msg_num':   decode_msg_type(full_payload),
                    'raw_msg':   combined_raw,
                    'vsi':       None
                })
                vdm_buffer = []
        elif msg.startswith('$AIVSI'):
            if records:
                records[-1]['vsi'] = msg

    return pd.DataFrame(records, columns=['timestamp','msg_num','raw_msg','vsi'])

print('함수 정의 완료')

함수 정의 완료


## 3. 시트별 데이터 품질 검증

In [7]:
for sheet_name in xl.sheet_names:
    df = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=None)
    df.columns = ['timestamp', 'raw_msg']

    issues = []
    buf = []

    for idx, row in df.iterrows():
        msg = str(row['raw_msg']).strip()
        if not msg.startswith('!'):
            continue
        parts    = msg.split(',')
        total    = int(parts[1])
        part_num = int(parts[2])

        if total == 1:
            if buf:
                issues.append({'row': idx, 'type': '멀티파트 미완성', 'msg': buf[0][1]})
                buf = []
        else:
            if part_num == 1:
                if buf:
                    issues.append({'row': idx, 'type': '멀티파트 미완성', 'msg': buf[0][1]})
                buf = [(idx, msg, part_num, total)]
            else:
                if not buf:
                    issues.append({'row': idx, 'type': '앞조각 없음', 'msg': msg})
                elif part_num != buf[-1][2] + 1:
                    issues.append({'row': idx, 'type': '순서 오류', 'msg': msg})
                else:
                    buf.append((idx, msg, part_num, total))
                if buf and part_num == total:
                    buf = []

    if buf:
        issues.append({'row': 'EOF', 'type': '멀티파트 미완성(파일 끝)', 'msg': buf[0][1]})

    status = f'문제 {len(issues)}건' if issues else '이상 없음'
    print(f'[{sheet_name}] {status}')

[26.06.09] 이상 없음
[26.06.10(~08 30)] 이상 없음
[26.06.10(10 33 ~ 15 06)] 이상 없음


## 4. 시트별 Type 1/3 파싱 및 xlsx 저장

In [8]:
import re
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 컬럼 그룹 정의 ──────────────────────────────────────────
COL_GROUPS = {
    'vdm':  {'cols': ['timestamp', 'vdm_channel'],
              'header_bg': '2E4D9F', 'header_font': 'FFFFFF'},
    'vsi':  {'cols': ['vsi_ui','vsi_link','vsi_hour','vsi_minute','vsi_second',
                       'vsi_slot','vsi_strength','vsi_snr'],
              'header_bg': '375623', 'header_font': 'FFFFFF'},
    'ais':  {'cols': ['msg_type','repeat','mmsi','status','turn','speed','accuracy',
                       'lon','lat','course','heading','second','maneuver','raim','radio'],
              'header_bg': '1F78B4', 'header_font': 'FFFFFF'},
    'comm': {'cols': ['sync_state','slot_timeout','sub_message',
                       'slot_increment','num_slots','keep_flag'],
              'header_bg': '00838F', 'header_font': 'FFFFFF'},
    'raw':  {'cols': ['raw_msg','vsi'],
              'header_bg': '595959', 'header_font': 'FFFFFF'},
}
COL_ORDER = (COL_GROUPS['vdm']['cols'] + COL_GROUPS['vsi']['cols'] +
             COL_GROUPS['ais']['cols'] + COL_GROUPS['comm']['cols'] +
             COL_GROUPS['raw']['cols'])

COL_TO_GROUP = {}
for g, info in COL_GROUPS.items():
    for c in info['cols']:
        COL_TO_GROUP[c] = g

ROW_FILL = {
    1: PatternFill('solid', fgColor='FFFDE7'),
    3: PatternFill('solid', fgColor='F3E5F5'),
}
NO_VSI_FILL = PatternFill('solid', fgColor='FFCCCC')  # VSI 없는 행: 빨간색

MSG_NAMES = {
    1:  'Position Report Class A (SOTDMA)',
    2:  'Position Report Class A (Assigned)',
    3:  'Position Report Class A (ITDMA)',
    4:  'Base Station Report',
    5:  'Static and Voyage Related Data',
    6:  'Addressed Binary Message',
    7:  'Binary Acknowledge',
    8:  'Binary Broadcast Message',
    12: 'Addressed Safety Message',
    15: 'Interrogation',
    18: 'Standard Class B CS Position Report',
    19: 'Extended Class B CS Position Report',
    20: 'Data Link Management',
    21: 'Aid-to-Navigation Report',
    24: 'Static Data Report',
    27: 'Long Range AIS Broadcast',
}

thin   = Side(style='thin', color='CCCCCC')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def apply_header(ws, columns):
    for col_idx, col_name in enumerate(columns, 1):
        g    = COL_TO_GROUP.get(col_name, 'raw')
        info = COL_GROUPS[g]
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill      = PatternFill('solid', fgColor=info['header_bg'])
        cell.font      = Font(bold=True, color=info['header_font'], size=10)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = border
    ws.row_dimensions[1].height = 30

def write_data_rows(ws, df, columns):
    for row_idx, record in enumerate(df.itertuples(index=False), 2):
        msg_type = getattr(record, 'msg_type', None)
        has_vsi  = getattr(record, 'vsi', None) is not None
        row_fill = NO_VSI_FILL if not has_vsi else ROW_FILL.get(msg_type)
        for col_idx, col_name in enumerate(columns, 1):
            val  = getattr(record, col_name, None)
            cell = ws.cell(row=row_idx, column=col_idx, value=val)
            cell.border    = border
            cell.alignment = Alignment(vertical='center')
            if row_fill:
                cell.fill = row_fill

def write_summary_sheet(ws, all_stats):
    ws.merge_cells('A1:F1')
    title = ws['A1']
    title.value     = 'AIS 메시지 수신 요약'
    title.font      = Font(bold=True, size=14, color='FFFFFF')
    title.fill      = PatternFill('solid', fgColor='2E4D9F')
    title.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 30

    cur_row     = 3
    grand_total = 0
    grand_no_vsi = 0

    for sheet_name, stats in all_stats.items():
        # 시트명 헤더
        ws.merge_cells(f'A{cur_row}:F{cur_row}')
        sh = ws.cell(row=cur_row, column=1, value=f'[ {sheet_name} ]')
        sh.font      = Font(bold=True, size=11, color='FFFFFF')
        sh.fill      = PatternFill('solid', fgColor='375623')
        sh.alignment = Alignment(horizontal='left', vertical='center')
        ws.row_dimensions[cur_row].height = 22
        cur_row += 1

        # 컬럼 헤더
        for col_idx, label in enumerate(['msg_num','메시지 이름','수신 개수','비율(%)','VSI 없는 수','VSI 없는 비율(%)'], 1):
            c = ws.cell(row=cur_row, column=col_idx, value=label)
            c.font      = Font(bold=True, color='FFFFFF')
            c.fill      = PatternFill('solid', fgColor='1F78B4')
            c.alignment = Alignment(horizontal='center', vertical='center')
            c.border    = border
        cur_row += 1

        sheet_total  = stats['total']
        sheet_no_vsi = stats['no_vsi_total']
        grand_total  += sheet_total
        grand_no_vsi += sheet_no_vsi

        for msg_num, count in sorted(stats['counts'].items()):
            name      = MSG_NAMES.get(msg_num, f'Type {msg_num}')
            ratio     = f'{count / sheet_total * 100:.1f}%' if sheet_total else '-'
            no_vsi_c  = stats['no_vsi_counts'].get(msg_num, 0)
            no_vsi_r  = f'{no_vsi_c / count * 100:.1f}%' if count else '-'
            fill      = ROW_FILL.get(msg_num, PatternFill('solid', fgColor='F5F5F5'))
            for col_idx, val in enumerate([msg_num, name, count, ratio, no_vsi_c, no_vsi_r], 1):
                c = ws.cell(row=cur_row, column=col_idx, value=val)
                c.fill      = fill
                c.alignment = Alignment(horizontal='center' if col_idx != 2 else 'left', vertical='center')
                c.border    = border
            cur_row += 1

        # 소계
        subtotal_no_vsi_r = f'{sheet_no_vsi / sheet_total * 100:.1f}%' if sheet_total else '-'
        for col_idx, val in enumerate(['소계', '', sheet_total, '', sheet_no_vsi, subtotal_no_vsi_r], 1):
            c = ws.cell(row=cur_row, column=col_idx, value=val)
            c.font   = Font(bold=True)
            c.fill   = PatternFill('solid', fgColor='E0E0E0')
            c.border = border
        cur_row += 2

    # 총합계
    total_no_vsi_r = f'{grand_no_vsi / grand_total * 100:.1f}%' if grand_total else '-'
    for col_idx, val in enumerate(['총 수신 메시지', '', grand_total, '', grand_no_vsi, total_no_vsi_r], 1):
        c = ws.cell(row=cur_row, column=col_idx, value=val)
        c.font   = Font(bold=True, color='FFFFFF')
        c.fill   = PatternFill('solid', fgColor='2E4D9F')
        c.border = border

    for col, width in zip('ABCDEF', [10, 42, 12, 10, 12, 16]):
        ws.column_dimensions[col].width = width


# ── 시트별 처리 ──────────────────────────────────────────────
all_stats = {}

for sheet_name in xl.sheet_names:
    print(f'처리 중: {sheet_name}')

    df_raw = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=None)
    df_raw.columns = ['timestamp', 'raw_msg']

    result_df = build_result_df(df_raw)

    # 통계 수집 (VSI 없는 메시지 포함)
    counts        = result_df['msg_num'].value_counts().sort_index().to_dict()
    no_vsi_mask   = result_df['vsi'].isna()
    no_vsi_counts = result_df[no_vsi_mask]['msg_num'].value_counts().sort_index().to_dict()
    all_stats[sheet_name] = {
        'total':         len(result_df),
        'counts':        counts,
        'no_vsi_total':  int(no_vsi_mask.sum()),
        'no_vsi_counts': no_vsi_counts,
    }

    # Type 1/3 파싱
    mask      = result_df['msg_num'].isin([1, 3])
    df_type13 = pd.DataFrame(result_df[mask].apply(parse_type_1_3, axis=1).tolist())
    for col in COL_ORDER:
        if col not in df_type13.columns:
            df_type13[col] = None
    df_type13 = df_type13[COL_ORDER]

    # 엑셀 저장
    safe_name   = re.sub(r'[\\/:*?"<>|]', '_', sheet_name)
    output_path = f'AIS_type13_{safe_name}.xlsx'

    wb         = Workbook()
    ws_summary = wb.active
    ws_summary.title = 'Summary'
    write_summary_sheet(ws_summary, {sheet_name: all_stats[sheet_name]})

    ws_data = wb.create_sheet(title='Type1_3')
    apply_header(ws_data, COL_ORDER)
    write_data_rows(ws_data, df_type13, COL_ORDER)
    for col_idx, col_name in enumerate(COL_ORDER, 1):
        ws_data.column_dimensions[get_column_letter(col_idx)].width = max(12, len(col_name) + 2)
    ws_data.freeze_panes = 'A2'

    wb.save(output_path)
    no_vsi_cnt = all_stats[sheet_name]['no_vsi_total']
    print(f'  저장 완료: {output_path} (Type1/3: {len(df_type13)}건, VSI 없음: {no_vsi_cnt}건)')

print('\n전체 완료')

처리 중: 26.06.09
  저장 완료: AIS_type13_26.06.09.xlsx (Type1/3: 314962건, VSI 없음: 0건)
처리 중: 26.06.10(~08 30)
  저장 완료: AIS_type13_26.06.10(~08 30).xlsx (Type1/3: 457453건, VSI 없음: 0건)
처리 중: 26.06.10(10 33 ~ 15 06)
  저장 완료: AIS_type13_26.06.10(10 33 ~ 15 06).xlsx (Type1/3: 162814건, VSI 없음: 0건)

전체 완료
